# DORAnet → enzyme hypotheses → DNA design

This notebook converts DORAnet-generated reaction networks into concise downstream design tables: parsed reactions, enzyme hypotheses, enzyme-candidate templates, and non-operational DNA design plans for expert review.

In [26]:
from pathlib import Path
import glob
import json
import os
import re
import textwrap
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rdkit import Chem

In [27]:
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'
resultsDir = os.path.join(dataDir, 'Results/combinedEbolaVirus_bestMACAW_allDB_generative/')
DORANETmoleculesDataDir = os.path.join(resultsDir, 'DORAnet/InoAdeGuoXan/doranet_output_gen3/')
DNADesignResultsDir = os.path.join(resultsDir, 'DNA_design/')
os.makedirs(DNADesignResultsDir, exist_ok=True)

print("DNADesignResultsDir :", DNADesignResultsDir)

DNADesignResultsDir : /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/


In [28]:
# Subdirectories inside DNA_design
reactionResultsDir = os.path.join(DNADesignResultsDir, "01_reactions/")
moleculeResultsDir = os.path.join(DNADesignResultsDir, "02_molecules/")
DNADesignResultsDir   = os.path.join(DNADesignResultsDir, "03_enzyme_mapping/")
routeResultsDir    = os.path.join(DNADesignResultsDir, "04_routes/")
sequenceResultsDir = os.path.join(DNADesignResultsDir, "05_uniprot_sequences/")
dnaResultsDir      = os.path.join(DNADesignResultsDir, "06_optimized_dna/")
handoffResultsDir  = os.path.join(DNADesignResultsDir, "07_webtool_handoff/")

# Create all directories
for dirPath in [
    DNADesignResultsDir,
    reactionResultsDir,
    moleculeResultsDir,
    DNADesignResultsDir,
    routeResultsDir,
    sequenceResultsDir,
    dnaResultsDir,
    handoffResultsDir,
]:
    os.makedirs(dirPath, exist_ok=True)

print("reactionResultsDir  :", reactionResultsDir)

reactionResultsDir  : /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/01_reactions/


## 1. Discover DORAnet JSON files

In [29]:
DORANETmoleculesDataDir = Path(DORANETmoleculesDataDir)
def starterNumFromName(dirName):
    match = re.search(r"starter_(\d+)", dirName)
    return int(match.group(1)) if match else -1

allStarterDirPaths = sorted(
    [p for p in DORANETmoleculesDataDir.glob("starter_*") if p.is_dir()],
    key=lambda p: starterNumFromName(p.name),
)

fileInfoList = []
missingJsonDirs = []

for starterDirPath in allStarterDirPaths:
    dirName = starterDirPath.name
    expectedJsonPath = starterDirPath / f"{dirName}_network_pretreated.json"

    if expectedJsonPath.exists():
        jsonPath = expectedJsonPath
    else:
        fallbackJsonPaths = sorted(starterDirPath.glob("*_network_pretreated.json"))
        jsonPath = fallbackJsonPaths[0] if fallbackJsonPaths else None

    if jsonPath is None:
        missingJsonDirs.append(dirName)
        continue

    fileInfoList.append({
        "dirName": dirName,
        "dirPath": str(starterDirPath),
        "jsonPath": str(jsonPath),
        "starterNum": starterNumFromName(dirName),
    })

print(f"Starter directories found : {len(allStarterDirPaths)}")
print(f"JSON files found          : {len(fileInfoList)}")
print(f"Missing JSON directories  : {len(missingJsonDirs)}")
fileInfoList[:3]

Starter directories found : 20
JSON files found          : 19
Missing JSON directories  : 1


[{'dirName': 'starter_00000',
  'dirPath': '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/doranet_output_gen3/starter_00000',
  'jsonPath': '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/doranet_output_gen3/starter_00000/starter_00000_gen3_network_pretreated.json',
  'starterNum': 0},
 {'dirName': 'starter_00001',
  'dirPath': '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/doranet_output_gen3/starter_00001',
  'jsonPath': '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/doranet_output_gen3/starter_00001/starter_00001_gen3_network_pretreated.json',
  'starterNum': 1},
 {'dirName': 'starter_00002',
  'dirPath': '/mnt

## 2. Parse DORAnet reaction strings

In [30]:
def splitMoleculeString(moleculeString):
    return [mol for mol in str(moleculeString).split(".") if mol]

def parseDoranetReaction(rxnString, fileInfo):
    parts = str(rxnString).split(">")
    if len(parts) != 4:
        raise ValueError(f"Expected 4 fields separated by '>'; found {len(parts)}")

    reactants, ruleName, metaBlock, products = parts
    metaParts = (metaBlock.split("$") + [None, None, None, None])[:4]
    thermo, reactantStoich, productStoich, reactionType = metaParts

    return {
        "reactants": reactants,
        "products": products,
        "reactionString": f"{reactants} >> {products}",
        "ruleName": ruleName,
        "thermo": thermo,
        "reactantStoich": reactantStoich,
        "productStoich": productStoich,
        "reactionType": reactionType,
        "numReactantMolecules": len(splitMoleculeString(reactants)),
        "numProductMolecules": len(splitMoleculeString(products)),
        "SourceStarterNum": fileInfo["starterNum"],
        "SourceDirectory": fileInfo["dirName"],
    }

reactionRecords = []
jsonReadErrors = []
uniqueReactantMolecules = set()
uniqueProductMolecules = set()

for fileInfo in tqdm(fileInfoList, desc="Reading DORAnet JSON"):
    try:
        with open(fileInfo["jsonPath"], "r", encoding="utf-8") as f:
            reactionList = json.load(f)

        for rxnString in reactionList:
            record = parseDoranetReaction(rxnString, fileInfo)
            reactionRecords.append(record)
            uniqueReactantMolecules.update(splitMoleculeString(record["reactants"]))
            uniqueProductMolecules.update(splitMoleculeString(record["products"]))

    except Exception as exc:
        jsonReadErrors.append({**fileInfo, "error": str(exc)})

if not reactionRecords:
    raise RuntimeError("No reactions loaded. Check DORANETmoleculesDataDir and JSON format.")

reactionDF = pd.DataFrame(reactionRecords).reset_index(drop=True)
reactionDF.to_csv(DNADesignResultsDir + "reactionDF.csv", index=False)

print(f"Reactions loaded     : {len(reactionDF):,}")
print(f"Failed JSON files    : {len(jsonReadErrors):,}")
print(f"Unique reactant mols : {len(uniqueReactantMolecules):,}")
print(f"Unique product mols  : {len(uniqueProductMolecules):,}")
reactionDF.head()

Reading DORAnet JSON: 100%|████████████████████████████████████████████████████████████████████████████████| 19/19 [00:00<00:00, 24.67it/s]


Reactions loaded     : 154,373
Failed JSON files    : 0
Unique reactant mols : 4,017
Unique product mols  : 63,776


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,SourceStarterNum,SourceDirectory
0,CCC(=O)OP(=O)(O)OC(C)O.O=P(O)(O)O,CC(O)O.CCC(=O)OP(=O)(O)OP(=O)(O)O,CCC(=O)OP(=O)(O)OC(C)O.O=P(O)(O)O >> CC(O)O.CC...,rule0768_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004
1,CC(=O)OP(=O)(O)OC(O)C(O)C(=O)O.CC(=O)OP(=O)(O)...,CC(=O)OP(=O)(O)OC(O)C(=O)C(=O)O.CC(=O)OP(=O)(O...,CC(=O)OP(=O)(O)OC(O)C(O)C(=O)O.CC(=O)OP(=O)(O)...,rule0324_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004
2,O=P1(O)OC(O)COC(CO)O1.O=P1(O)OC(O)COC(CO)O1,O=P1(O)OC(CO)OCC(OC(O)CO)O1.O=P1(O)OCC(O)O1,O=P1(O)OC(O)COC(CO)O1.O=P1(O)OC(O)COC(CO)O1 >>...,rule0384_1,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004
3,CC(=O)OP(=O)(O)OC(O)C(C)O.NC(=O)c1ccc[n+]([C@@...,CC(=O)OP(=O)(O)OC(=O)C(C)O.NC(=O)C1=CN([C@@H]2...,CC(=O)OP(=O)(O)OC(O)C(C)O.NC(=O)c1ccc[n+]([C@@...,rule0002_144,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004
4,CCC(=O)OP(=O)(O)OC(=O)C(=O)O.OO,CCC(=O)OP(=O)(O)OC(=O)C(O)O.O=O,CCC(=O)OP(=O)(O)OC(=O)C(=O)O.OO >> CCC(=O)OP(=...,rule0078_15,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004


### Keep only enzymatic DORAnet reactions

In [32]:
enzymaticReactionDF = reactionDF[
    reactionDF["reactionType"].astype(str).str.lower().str.contains(
        "enzyme|enzymatic|bio|biological",
        na=False
    )
].copy()

print(f"Total reactions: {len(reactionDF):,}")
print(f"Likely enzymatic reactions: {len(enzymaticReactionDF):,}")

enzymaticReactionDF.head()

Total reactions: 154,373
Likely enzymatic reactions: 154,373


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,SourceStarterNum,SourceDirectory
0,CCC(=O)OP(=O)(O)OC(C)O.O=P(O)(O)O,CC(O)O.CCC(=O)OP(=O)(O)OP(=O)(O)O,CCC(=O)OP(=O)(O)OC(C)O.O=P(O)(O)O >> CC(O)O.CC...,rule0768_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004
1,CC(=O)OP(=O)(O)OC(O)C(O)C(=O)O.CC(=O)OP(=O)(O)...,CC(=O)OP(=O)(O)OC(O)C(=O)C(=O)O.CC(=O)OP(=O)(O...,CC(=O)OP(=O)(O)OC(O)C(O)C(=O)O.CC(=O)OP(=O)(O)...,rule0324_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004
2,O=P1(O)OC(O)COC(CO)O1.O=P1(O)OC(O)COC(CO)O1,O=P1(O)OC(CO)OCC(OC(O)CO)O1.O=P1(O)OCC(O)O1,O=P1(O)OC(O)COC(CO)O1.O=P1(O)OC(O)COC(CO)O1 >>...,rule0384_1,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004
3,CC(=O)OP(=O)(O)OC(O)C(C)O.NC(=O)c1ccc[n+]([C@@...,CC(=O)OP(=O)(O)OC(=O)C(C)O.NC(=O)C1=CN([C@@H]2...,CC(=O)OP(=O)(O)OC(O)C(C)O.NC(=O)c1ccc[n+]([C@@...,rule0002_144,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004
4,CCC(=O)OP(=O)(O)OC(=O)C(=O)O.OO,CCC(=O)OP(=O)(O)OC(=O)C(O)O.O=O,CCC(=O)OP(=O)(O)OC(=O)C(=O)O.OO >> CCC(=O)OP(=...,rule0078_15,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004


### Summarize unique DORAnet rules

Many reactions may use the same rule. We do not want to annotate millions of reactions one by one. First annotate the rules.

In [49]:
ruleSummaryDF = (
    enzymaticReactionDF
    .groupby(["ruleName", "reactionType"], dropna=False)
    .agg(
        numReactions=("reactionString", "count"),
        numStarterSources=("SourceStarterNum", "nunique"),
        exampleReaction=("reactionString", "first"),
        exampleReactants=("reactants", "first"),
        exampleProducts=("products", "first"),
    )
    .reset_index()
    .sort_values("numReactions", ascending=False)
)

print(f"unique DORAnet rules    : {reactionDF['ruleName'].nunique():,}")
ruleSummaryDF

unique DORAnet rules    : 567


,ruleName,reactionType,numReactions,numStarterSources,exampleReaction,exampleReactants,exampleProducts
483,rule0169_1,Enzymatic,28752,13,CC(=O)OP(=O)(O)OC(=O)COC(=O)CO >> CC(O)OP1(=O)...,CC(=O)OP(=O)(O)OC(=O)COC(=O)CO,CC(O)OP1(=O)OC(=O)COC(=O)C(O)O1
461,rule0126_2,Enzymatic,10969,12,CCC(=O)OP(=O)(O)OC(O)CO.CCC(=O)OP(=O)(O)OC(O)C...,CCC(=O)OP(=O)(O)OC(O)CO.CCC(=O)OP(=O)(O)OC(O)CO,CC(=O)OP(=O)(O)OC(O)CO.CCC(=O)OP(=O)(O)OC(C)(O)CO
332,rule0028_51,Enzymatic,7942,12,CC(O)OP(=O)(O)OC(O)CO >> CC(O)(CO)OP(=O)(O)OCO,CC(O)OP(=O)(O)OC(O)CO,CC(O)(CO)OP(=O)(O)OCO
331,rule0028_50,Enzymatic,5529,12,CC(=O)OP(=O)(O)OC(=O)C(O)C(=O)O >> O=C(O)CC(=O...,CC(=O)OP(=O)(O)OC(=O)C(O)C(=O)O,O=C(O)CC(=O)OP(=O)(O)OC(=O)CO
395,rule0070_6,Enzymatic,4733,12,CC(=O)OP(=O)(O)OC(=O)CC(=O)O.O >> O=C(CO)OP(=O...,CC(=O)OP(=O)(O)OC(=O)CC(=O)O.O,O=C(CO)OP(=O)(O)OC(=O)CC(O)O
...,...,...,...,...,...,...,...
58,rule0003_086,Enzymatic,1,1,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...
73,rule0003_127,Enzymatic,1,1,CC(C)C(=O)C(=O)O.NC(=O)C1=CN([C@@H]2O[C@H](COP...,CC(C)C(=O)C(=O)O.NC(=O)C1=CN([C@@H]2O[C@H](COP...,CC(C)C(O)C(=O)O.NC(=O)c1ccc[n+]([C@@H]2O[C@H](...
466,rule0137_3,Enzymatic,1,1,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc...
482,rule0168_5,Enzymatic,1,1,O=CNc1ncnc2c1ncn2[C@@H]1O[C@@](O)(CO)[C@@H](O)...,O=CNc1ncnc2c1ncn2[C@@H]1O[C@@](O)(CO)[C@@H](O)...,O=CNc1ncnc2c1ncn2[C@H](O)[C@H](O)[C@H](O)[C@H]...


### Add required placeholder columns

In [50]:
enzymeAnnotationDF = ruleSummaryDF.copy()

enzymeAnnotationDF["suggestedEnzymeClass"] = ""
enzymeAnnotationDF["ecNumber"] = ""
enzymeAnnotationDF["enzymeName"] = ""
enzymeAnnotationDF["uniprotAccession"] = ""
enzymeAnnotationDF["sourceDatabase"] = ""
enzymeAnnotationDF["reactionSimilarity"] = ""
enzymeAnnotationDF["enzymeConfidence"] = ""
enzymeAnnotationDF["notes"] = ""

enzymeAnnotationDF.to_csv(os.path.join(DNADesignResultsDir, "enzymeAnnotationTemplate.csv"),index=False)
enzymeAnnotationDF.head()

,ruleName,reactionType,numReactions,numStarterSources,exampleReaction,exampleReactants,exampleProducts,suggestedEnzymeClass,ecNumber,enzymeName,uniprotAccession,sourceDatabase,reactionSimilarity,enzymeConfidence,notes
483,rule0169_1,Enzymatic,28752,13,CC(=O)OP(=O)(O)OC(=O)COC(=O)CO >> CC(O)OP1(=O)...,CC(=O)OP(=O)(O)OC(=O)COC(=O)CO,CC(O)OP1(=O)OC(=O)COC(=O)C(O)O1,,,,,,,,
461,rule0126_2,Enzymatic,10969,12,CCC(=O)OP(=O)(O)OC(O)CO.CCC(=O)OP(=O)(O)OC(O)C...,CCC(=O)OP(=O)(O)OC(O)CO.CCC(=O)OP(=O)(O)OC(O)CO,CC(=O)OP(=O)(O)OC(O)CO.CCC(=O)OP(=O)(O)OC(C)(O)CO,,,,,,,,
332,rule0028_51,Enzymatic,7942,12,CC(O)OP(=O)(O)OC(O)CO >> CC(O)(CO)OP(=O)(O)OCO,CC(O)OP(=O)(O)OC(O)CO,CC(O)(CO)OP(=O)(O)OCO,,,,,,,,
331,rule0028_50,Enzymatic,5529,12,CC(=O)OP(=O)(O)OC(=O)C(O)C(=O)O >> O=C(O)CC(=O...,CC(=O)OP(=O)(O)OC(=O)C(O)C(=O)O,O=C(O)CC(=O)OP(=O)(O)OC(=O)CO,,,,,,,,
395,rule0070_6,Enzymatic,4733,12,CC(=O)OP(=O)(O)OC(=O)CC(=O)O.O >> O=C(CO)OP(=O...,CC(=O)OP(=O)(O)OC(=O)CC(=O)O.O,O=C(CO)OP(=O)(O)OC(=O)CC(O)O,,,,,,,,


### Read DORAnet reaction ruleset file

In [51]:
doranetRulesetDF = pd.read_csv(dataDir + "/DORAnet/JN3604IMT_rules.tsv", sep="\t")

print(doranetRulesetDF.shape)
print(doranetRulesetDF.columns.tolist())
doranetRulesetDF.head()

(3604, 5)
['Name', 'Reactants', 'SMARTS', 'Products', 'Comments']


,Name,Reactants,SMARTS,Products,Comments
0,rule0001_01,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,P97259;P97402;Q08834;Q09328;Q16408;Q3V5L5;Q765...
1,rule0001_02,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,A0K8D0;A0KHH6;A0KZ10;A0L8R9;A0Q7X9;A1A7M6;A1K6...
2,rule0001_03,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,P0DMP6;P0DMP7;P16442;Q15951;Q1RLK6;Q3USF0;Q5HZ...
3,rule0001_04,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,A4IID1;O19071;O77836;P23336;P26572;P27115;P278...
4,rule0001_05,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,NaN


In [52]:
reactionRuleSet = set(
    reactionDF["ruleName"]
    .dropna()
    .astype(str)
    .str.strip()
)

tsvRuleSet = set(
    doranetRulesetDF["Name"]
    .dropna()
    .astype(str)
    .str.strip()
)

matchedRuleSet = reactionRuleSet.intersection(tsvRuleSet)
missingRuleSet = reactionRuleSet.difference(tsvRuleSet)

print(f"Rules in reactionDF             : {len(reactionRuleSet):,}")
print(f"Rules matched in JN3604IMT TSV  : {len(matchedRuleSet):,}")
print(f"Rules missing from JN3604IMT TSV: {len(missingRuleSet):,}")

print("\nExample matched rules:")
print(sorted(list(matchedRuleSet))[:20])

print("\nExample missing rules:")
print(sorted(list(missingRuleSet))[:20])

Rules in reactionDF             : 567
Rules matched in JN3604IMT TSV  : 567
Rules missing from JN3604IMT TSV: 0

Example matched rules:
['rule0001_67', 'rule0001_79', 'rule0001_81', 'rule0001_86', 'rule0001_87', 'rule0001_88', 'rule0001_89', 'rule0001_90', 'rule0002_045', 'rule0002_068', 'rule0002_069', 'rule0002_071', 'rule0002_079', 'rule0002_080', 'rule0002_085', 'rule0002_089', 'rule0002_093', 'rule0002_094', 'rule0002_095', 'rule0002_096']

Example missing rules:
[]


In [40]:
targetRuleNameList = [
    "rule0169_1",
    "rule0126_2",
    "rule0028_51",
    "rule0028_50",
    "rule0070_6",
]

selectedRuleDF = doranetRulesetDF[
    doranetRulesetDF["Name"].isin(targetRuleNameList)
].copy()

selectedRuleDF = selectedRuleDF.rename(columns={
    "Name": "ruleName",
    "Reactants": "ruleReactants",
    "SMARTS": "ruleSMARTS",
    "Products": "ruleProducts",
    "Comments": "candidateUniProtRaw",
})

def splitUniProtIds(idString):
    if pd.isna(idString):
        return []
    return [x.strip() for x in str(idString).split(";") if x.strip()]

selectedRuleDF["candidateUniProtList"] = selectedRuleDF["candidateUniProtRaw"].apply(splitUniProtIds)
selectedRuleDF["numCandidateUniProt"] = selectedRuleDF["candidateUniProtList"].apply(len)
selectedRuleDF["first10UniProtIds"] = selectedRuleDF["candidateUniProtList"].apply(lambda x: ";".join(x[:10]))

selectedRuleDF[
    [
        "ruleName",
        "ruleReactants",
        "ruleProducts",
        "numCandidateUniProt",
        "first10UniProtIds",
        "ruleSMARTS",
    ]
]
selectedRuleDF

,ruleName,ruleReactants,ruleSMARTS,ruleProducts,candidateUniProtRaw,candidateUniProtList,numCandidateUniProt,first10UniProtIds
2091,rule0028_50,Any,([#6;!$([#6&!R](-&!@[#6&!R](-&!@[#6&!R](-&!@[#...,Any,A0A1D3TPC3;A0A1D3TXG7;A0AKV8;A0ALE0;A0B6L2;A0J...,"[A0A1D3TPC3, A0A1D3TXG7, A0AKV8, A0ALE0, A0B6L...",827,A0A1D3TPC3;A0A1D3TXG7;A0AKV8;A0ALE0;A0B6L2;A0J...
2092,rule0028_51,Any,([#6;!$([#6&!R]-&!@[#6&!R](-&!@[#6&!R]-&!@[#6&...,Any,A8YZE2,[A8YZE2],1,A8YZE2
2645,rule0070_6,Any;WATER,([#6;$([#6&!R]-&!@[#6&!R]);!$([#6&!R]-&!@[#6&!...,Any,A0AG18;A0JUZ5;A0K3V4;A0LBT6;A0LQG8;A0LTS3;A0PP...,"[A0AG18, A0JUZ5, A0K3V4, A0LBT6, A0LQG8, A0LTS...",706,A0AG18;A0JUZ5;A0K3V4;A0LBT6;A0LQG8;A0LTS3;A0PP...
2963,rule0126_2,Any;Any,[#6;!$([#6&!R]-[#6&!R]-&!@[#6&!R](=&!@[#8&!R])...,Any;Any,A1ADQ1;A4YXN2;A5EGD7;A6QP15;A7ZPI2;A8A2M8;A8WT...,"[A1ADQ1, A4YXN2, A5EGD7, A6QP15, A7ZPI2, A8A2M...",72,A1ADQ1;A4YXN2;A5EGD7;A6QP15;A7ZPI2;A8A2M8;A8WT...
3047,rule0169_1,Any,([#6:1].[#8;$([#8&!R]):2].[#6;$([#6&!R]=&!@[#8...,Any,A0A0H4BJU5;A0AF79;A0AJ81;A0AK66;A0B9D8;A0JRF5;...,"[A0A0H4BJU5, A0AF79, A0AJ81, A0AK66, A0B9D8, A...",1602,A0A0H4BJU5;A0AF79;A0AJ81;A0AK66;A0B9D8;A0JRF5;...


### Merge rule information into `reactionDF`

This connects DORAnet reactions with the rule SMARTS and UniProt candidate list.

In [42]:
doranetRulesetDF = selectedRuleDF[
    [
        "ruleName",
        "ruleReactants",
        "ruleProducts",
        "ruleSMARTS",
        "candidateUniProtRaw",
        "numCandidateUniProt",
    ]
].copy()

reactionWithRuleDF = reactionDF.merge(
    doranetRulesetDF,
    on="ruleName",
    how="left"
)

reactionWithRuleDF["hasRuleLookup"] = reactionWithRuleDF["ruleSMARTS"].notna()

reactionWithRuleDF.to_csv(
    os.path.join(DNADesignResultsDir, "reactionWithRuleDF.csv"),
    index=False
)

print(reactionWithRuleDF["hasRuleLookup"].value_counts(dropna=False))

reactionWithRuleDF[
    [
        "reactionString",
        "ruleName",
        "reactionType",
        "ruleReactants",
        "ruleProducts",
        "numCandidateUniProt",
    ]
].head()

hasRuleLookup
False    96448
True     57925
Name: count, dtype: int64


,reactionString,ruleName,reactionType,ruleReactants,ruleProducts,numCandidateUniProt
0,CCC(=O)OP(=O)(O)OC(C)O.O=P(O)(O)O >> CC(O)O.CC...,rule0768_2,Enzymatic,NaN,NaN,NaN
1,CC(=O)OP(=O)(O)OC(O)C(O)C(=O)O.CC(=O)OP(=O)(O)...,rule0324_2,Enzymatic,NaN,NaN,NaN
2,O=P1(O)OC(O)COC(CO)O1.O=P1(O)OC(O)COC(CO)O1 >>...,rule0384_1,Enzymatic,NaN,NaN,NaN
3,CC(=O)OP(=O)(O)OC(O)C(C)O.NC(=O)c1ccc[n+]([C@@...,rule0002_144,Enzymatic,NaN,NaN,NaN
4,CCC(=O)OP(=O)(O)OC(=O)C(=O)O.OO >> CCC(=O)OP(=...,rule0078_15,Enzymatic,NaN,NaN,NaN


In [45]:
doranetRulesetDF.columns

Index(['ruleName', 'ruleReactants', 'ruleProducts', 'ruleSMARTS',
       'candidateUniProtRaw', 'numCandidateUniProt'],
      dtype='object')

In [46]:
reactionRuleSet = set(reactionDF["ruleName"].dropna().astype(str).str.strip())
tsvRuleSet = set(doranetRulesetDF["ruleName"].dropna().astype(str).str.strip())

matchedRuleSet = reactionRuleSet.intersection(tsvRuleSet)
missingRuleSet = reactionRuleSet.difference(tsvRuleSet)

print(f"Rules in reactionDF             : {len(reactionRuleSet):,}")
print(f"Rules matched in JN3604IMT TSV  : {len(matchedRuleSet):,}")
print(f"Rules missing from JN3604IMT TSV: {len(missingRuleSet):,}")

print("\nExample matched rules:")
print(sorted(list(matchedRuleSet))[:20])

print("\nExample missing rules:")
print(sorted(list(missingRuleSet))[:20])

Rules in reactionDF             : 567
Rules matched in JN3604IMT TSV  : 5
Rules missing from JN3604IMT TSV: 562

Example matched rules:
['rule0028_50', 'rule0028_51', 'rule0070_6', 'rule0126_2', 'rule0169_1']

Example missing rules:
['rule0001_67', 'rule0001_79', 'rule0001_81', 'rule0001_86', 'rule0001_87', 'rule0001_88', 'rule0001_89', 'rule0001_90', 'rule0002_045', 'rule0002_068', 'rule0002_069', 'rule0002_071', 'rule0002_079', 'rule0002_080', 'rule0002_085', 'rule0002_089', 'rule0002_093', 'rule0002_094', 'rule0002_095', 'rule0002_096']


### Standardize `DORAnet` results with unique reaction IDs and canonicalized SMILES

In [16]:
def splitMoleculeString(moleculeString):
    return [mol for mol in str(moleculeString).split(".") if mol]


def canonicalizeSmiles(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)


def canonicalizeMoleculeString(moleculeString):
    canonicalMolList = []
    invalidMolList = []

    for molString in splitMoleculeString(moleculeString):
        canonicalMol = canonicalizeSmiles(molString)
        if canonicalMol is None:
            invalidMolList.append(molString)
        else:
            canonicalMolList.append(canonicalMol)

    canonicalMolList = sorted(canonicalMolList)

    return ".".join(canonicalMolList), invalidMolList


cleanReactionDF = reactionDF.copy().reset_index(drop=True)

cleanReactionDF["reactionId"] = [
    f"rxn_{idx:08d}" for idx in range(len(cleanReactionDF))
]

cleanReactionDF[["canonicalReactants", "invalidReactants"]] = cleanReactionDF["reactants"].apply(
    lambda x: pd.Series(canonicalizeMoleculeString(x))
)

cleanReactionDF[["canonicalProducts", "invalidProducts"]] = cleanReactionDF["products"].apply(
    lambda x: pd.Series(canonicalizeMoleculeString(x))
)

cleanReactionDF["isValidReactants"] = cleanReactionDF["invalidReactants"].apply(lambda x: len(x) == 0)
cleanReactionDF["isValidProducts"] = cleanReactionDF["invalidProducts"].apply(lambda x: len(x) == 0)
cleanReactionDF["isValidReaction"] = cleanReactionDF["isValidReactants"] & cleanReactionDF["isValidProducts"]

cleanReactionDF.to_csv(os.path.join(reactionResultsDir, "cleanReactionDF.csv"), index=False)

print(f"Total reactions        : {len(cleanReactionDF):,}")
print(f"Valid reactions        : {cleanReactionDF['isValidReaction'].sum():,}")
print(f"Invalid reactions      : {(~cleanReactionDF['isValidReaction']).sum():,}")

cleanReactionDF

Total reactions        : 154,373
Valid reactions        : 154,373
Invalid reactions      : 0


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,SourceStarterNum,SourceDirectory,reactionId,canonicalReactants,invalidReactants,canonicalProducts,invalidProducts,isValidReactants,isValidProducts,isValidReaction
0,CCC(=O)OP(=O)(O)OC(C)O.O=P(O)(O)O,CC(O)O.CCC(=O)OP(=O)(O)OP(=O)(O)O,CCC(=O)OP(=O)(O)OC(C)O.O=P(O)(O)O >> CC(O)O.CC...,rule0768_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004,rxn_00000000,CCC(=O)OP(=O)(O)OC(C)O.O=P(O)(O)O,[],CC(O)O.CCC(=O)OP(=O)(O)OP(=O)(O)O,[],True,True,True
1,CC(=O)OP(=O)(O)OC(O)C(O)C(=O)O.CC(=O)OP(=O)(O)...,CC(=O)OP(=O)(O)OC(O)C(=O)C(=O)O.CC(=O)OP(=O)(O...,CC(=O)OP(=O)(O)OC(O)C(O)C(=O)O.CC(=O)OP(=O)(O)...,rule0324_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004,rxn_00000001,CC(=O)OP(=O)(O)OC(O)C(O)C(=O)O.CC(=O)OP(=O)(O)...,[],CC(=O)OP(=O)(O)OC(O)C(=O)C(=O)O.CC(=O)OP(=O)(O...,[],True,True,True
2,O=P1(O)OC(O)COC(CO)O1.O=P1(O)OC(O)COC(CO)O1,O=P1(O)OC(CO)OCC(OC(O)CO)O1.O=P1(O)OCC(O)O1,O=P1(O)OC(O)COC(CO)O1.O=P1(O)OC(O)COC(CO)O1 >>...,rule0384_1,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004,rxn_00000002,O=P1(O)OC(O)COC(CO)O1.O=P1(O)OC(O)COC(CO)O1,[],O=P1(O)OC(CO)OCC(OC(O)CO)O1.O=P1(O)OCC(O)O1,[],True,True,True
3,CC(=O)OP(=O)(O)OC(O)C(C)O.NC(=O)c1ccc[n+]([C@@...,CC(=O)OP(=O)(O)OC(=O)C(C)O.NC(=O)C1=CN([C@@H]2...,CC(=O)OP(=O)(O)OC(O)C(C)O.NC(=O)c1ccc[n+]([C@@...,rule0002_144,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004,rxn_00000003,CC(=O)OP(=O)(O)OC(O)C(C)O.NC(=O)c1ccc[n+]([C@@...,[],CC(=O)OP(=O)(O)OC(=O)C(C)O.NC(=O)C1=CN([C@@H]2...,[],True,True,True
4,CCC(=O)OP(=O)(O)OC(=O)C(=O)O.OO,CCC(=O)OP(=O)(O)OC(=O)C(O)O.O=O,CCC(=O)OP(=O)(O)OC(=O)C(=O)O.OO >> CCC(=O)OP(=...,rule0078_15,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,4,starter_00004,rxn_00000004,CCC(=O)OP(=O)(O)OC(=O)C(=O)O.OO,[],CCC(=O)OP(=O)(O)OC(=O)C(O)O.O=O,[],True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154368,CO[C@@H]1C(C(O)O)=CO[C@H]1n1cnc2c(=O)[nH]c(N)n...,CO[C@@H]1C(C(=O)O)=CO[C@H]1n1cnc2c(=O)[nH]c(N)...,CO[C@@H]1C(C(O)O)=CO[C@H]1n1cnc2c(=O)[nH]c(N)n...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,19,starter_00019,rxn_00154368,CO[C@@H]1C(C(O)O)=CO[C@H]1n1cnc2c(=O)[nH]c(N)n...,[],CO[C@@H]1C(C(=O)O)=CO[C@H]1n1cnc2c(=O)[nH]c(N)...,[],True,True,True
154369,CC(=O)OC(O)C1=C[C@@H](O)[C@H](n2cnc3c4nc(nc32)...,CC(=O)OC(O)C1=C[C@](C)(O)[C@H](n2cnc3c4nc(nc32...,CC(=O)OC(O)C1=C[C@@H](O)[C@H](n2cnc3c4nc(nc32)...,rule0126_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,19,starter_00019,rxn_00154369,CC(=O)OC(O)C1=C[C@@H](O)[C@H](n2cnc3c4nc(nc32)...,[],CC(=O)OC(O)C1=C[C@](C)(O)[C@H](n2cnc3c4nc(nc32...,[],True,True,True
154370,CC(O)Nc1nc2c(ncn2[C@@H]2OC(C(O)O)=C[C@H]2O)c(=...,CCC(O)Nc1nc2c(ncn2[C@@H]2OC(C(O)O)=C[C@H]2O)c(...,CC(O)Nc1nc2c(ncn2[C@@H]2OC(C(O)O)=C[C@H]2O)c(=...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,19,starter_00019,rxn_00154370,CC(O)Nc1nc2c(ncn2[C@@H]2OC(C(O)O)=C[C@H]2O)c(=...,[],CCC(O)Nc1nc2c(ncn2[C@@H]2OC(C(O)O)=C[C@H]2O)c(...,[],True,True,True
154371,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,rule0062_18,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,19,starter_00019,rxn_00154371,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,[],CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc...,[],True,True,True


### Extract all unique molecules from reactants and products to `moleculeDF` data frame

In [17]:
moleculeRecords = []

for _, row in cleanReactionDF.iterrows():
    for mol in splitMoleculeString(row["canonicalReactants"]):
        moleculeRecords.append({
            "canonicalSmiles": mol,
            "role": "reactant",
            "reactionId": row["reactionId"],
            "ruleName": row["ruleName"],
            "SourceStarterNum": row["SourceStarterNum"],
            "SourceDirectory": row["SourceDirectory"],
        })

    for mol in splitMoleculeString(row["canonicalProducts"]):
        moleculeRecords.append({
            "canonicalSmiles": mol,
            "role": "product",
            "reactionId": row["reactionId"],
            "ruleName": row["ruleName"],
            "SourceStarterNum": row["SourceStarterNum"],
            "SourceDirectory": row["SourceDirectory"],
        })

moleculeRoleDF = pd.DataFrame(moleculeRecords)

moleculeDF = (
    moleculeRoleDF
    .groupby("canonicalSmiles")
    .agg(
        numAppearances=("reactionId", "count"),
        numReactions=("reactionId", "nunique"),
        numAsReactant=("role", lambda x: (x == "reactant").sum()),
        numAsProduct=("role", lambda x: (x == "product").sum()),
        numStarterSources=("SourceStarterNum", "nunique"),
        exampleRule=("ruleName", "first"),
    )
    .reset_index()
)

moleculeDF["isOnlyReactant"] = (moleculeDF["numAsReactant"] > 0) & (moleculeDF["numAsProduct"] == 0)
moleculeDF["isOnlyProduct"] = (moleculeDF["numAsProduct"] > 0) & (moleculeDF["numAsReactant"] == 0)
moleculeDF["isIntermediate"] = (moleculeDF["numAsReactant"] > 0) & (moleculeDF["numAsProduct"] > 0)

moleculeDF.to_csv(os.path.join(reactionResultsDir, "moleculeDF.csv"), index=False)
moleculeRoleDF.to_csv(os.path.join(reactionResultsDir, "moleculeRoleDF.csv"), index=False)

print(f"Unique molecules: {len(moleculeDF):,}")
moleculeDF

Unique molecules: 63,778


,canonicalSmiles,numAppearances,numReactions,numAsReactant,numAsProduct,numStarterSources,exampleRule,isOnlyReactant,isOnlyProduct,isIntermediate
0,*C1=C(*)C(=O)C(*)=C(*)C1=O,1497,1497,1443,54,12,rule0467_8,False,False,True
1,*c1c(*)c(O)c(*)c(*)c1O,1497,1497,54,1443,12,rule0467_8,False,False,True
2,C,555,555,0,555,12,rule0126_2,False,True,False
3,C#N,1083,1083,1020,63,7,rule0393_9,False,False,True
4,C1CO1,2,2,0,2,1,rule0053_20,False,True,False
...,...,...,...,...,...,...,...,...,...,...
63773,O[C@H]1[C@H]2Oc3nc4c5ncn(c5n3)[C@@H]2O[C@@H]1C=N4,1,1,0,1,1,rule0146_1,False,True,False
63774,O[C@H]1[C@H]2Oc3nc4c5ncnc4n3[C@@H]2O[C@@H]1C=N5,1,1,0,1,1,rule0146_1,False,True,False
63775,O[C@]12O[C@H]1[C@H]1O[C@@H]2C=Nc2ncnc3c2ncn31,1,1,0,1,1,rule0146_1,False,True,False
63776,Oc1ccc(O)c(O)c1,1,1,0,1,1,rule0308_1,False,True,False


## 3. Summarize `DORAnet` reaction rules

In [19]:
ruleSummaryDF = (
    cleanReactionDF
    .groupby(["ruleName", "reactionType"], dropna=False)
    .agg(
        numReactions=("reactionId", "count"),
        numStarterSources=("SourceStarterNum", "nunique"),
        exampleReaction=("reactionString", "first"),
        exampleReactants=("canonicalReactants", "first"),
        exampleProducts=("canonicalProducts", "first"),
    )
    .reset_index()
    .sort_values("numReactions", ascending=False)
)

ruleSummaryDF["enzymeName"] = ""
ruleSummaryDF["ecNumber"] = ""
ruleSummaryDF["uniprotAccession"] = ""
ruleSummaryDF["organism"] = ""
ruleSummaryDF["cofactor"] = ""
ruleSummaryDF["enzymeConfidence"] = ""
ruleSummaryDF["notes"] = ""

ruleSummaryDF.to_csv(os.path.join(reactionResultsDir, "enzymeAnnotationTemplate.csv"), index=False)

print(f"Unique DORAnet rules: {len(ruleSummaryDF):,}")
ruleSummaryDF

Unique DORAnet rules: 567


,ruleName,reactionType,numReactions,numStarterSources,exampleReaction,exampleReactants,exampleProducts,enzymeName,ecNumber,uniprotAccession,organism,cofactor,enzymeConfidence,notes
483,rule0169_1,Enzymatic,28752,13,CC(=O)OP(=O)(O)OC(=O)COC(=O)CO >> CC(O)OP1(=O)...,CC(=O)OP(=O)(O)OC(=O)COC(=O)CO,CC(O)OP1(=O)OC(=O)COC(=O)C(O)O1,,,,,,,
461,rule0126_2,Enzymatic,10969,12,CCC(=O)OP(=O)(O)OC(O)CO.CCC(=O)OP(=O)(O)OC(O)C...,CCC(=O)OP(=O)(O)OC(O)CO.CCC(=O)OP(=O)(O)OC(O)CO,CC(=O)OP(=O)(O)OC(O)CO.CCC(=O)OP(=O)(O)OC(C)(O)CO,,,,,,,
332,rule0028_51,Enzymatic,7942,12,CC(O)OP(=O)(O)OC(O)CO >> CC(O)(CO)OP(=O)(O)OCO,CC(O)OP(=O)(O)OC(O)CO,CC(O)(CO)OP(=O)(O)OCO,,,,,,,
331,rule0028_50,Enzymatic,5529,12,CC(=O)OP(=O)(O)OC(=O)C(O)C(=O)O >> O=C(O)CC(=O...,CC(=O)OP(=O)(O)OC(=O)C(O)C(=O)O,O=C(O)CC(=O)OP(=O)(O)OC(=O)CO,,,,,,,
395,rule0070_6,Enzymatic,4733,12,CC(=O)OP(=O)(O)OC(=O)CC(=O)O.O >> O=C(CO)OP(=O...,CC(=O)OP(=O)(O)OC(=O)CC(=O)O.O,O=C(CO)OP(=O)(O)OC(=O)CC(O)O,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,rule0003_086,Enzymatic,1,1,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O...,NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,,,,,,,
73,rule0003_127,Enzymatic,1,1,CC(C)C(=O)C(=O)O.NC(=O)C1=CN([C@@H]2O[C@H](COP...,CC(C)C(=O)C(=O)O.NC(=O)C1=CN([C@@H]2O[C@H](COP...,CC(C)C(O)C(=O)O.NC(=O)c1ccc[n+]([C@@H]2O[C@H](...,,,,,,,
466,rule0137_3,Enzymatic,1,1,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cnc...,,,,,,,
482,rule0168_5,Enzymatic,1,1,O=CNc1ncnc2c1ncn2[C@@H]1O[C@@](O)(CO)[C@@H](O)...,O=CNc1ncnc2c1ncn2[C@@H]1O[C@@](O)(CO)[C@@H](O)...,O=CNc1ncnc2c1ncn2[C@H](O)[C@H](O)[C@H](O)[C@H]...,,,,,,,


### 4. Build reaction graph

In [20]:
import networkx as nx

reactionGraph = nx.DiGraph()

validReactionDF = cleanReactionDF[cleanReactionDF["isValidReaction"]].copy()

for _, row in validReactionDF.iterrows():
    reactionNode = f"reaction::{row['reactionId']}"

    reactionGraph.add_node(
        reactionNode,
        nodeType="reaction",
        reactionId=row["reactionId"],
        ruleName=row["ruleName"],
        reactionType=row["reactionType"],
    )

    for reactantMol in splitMoleculeString(row["canonicalReactants"]):
        moleculeNode = f"molecule::{reactantMol}"
        reactionGraph.add_node(
            moleculeNode,
            nodeType="molecule",
            canonicalSmiles=reactantMol,
        )
        reactionGraph.add_edge(moleculeNode, reactionNode)

    for productMol in splitMoleculeString(row["canonicalProducts"]):
        moleculeNode = f"molecule::{productMol}"
        reactionGraph.add_node(
            moleculeNode,
            nodeType="molecule",
            canonicalSmiles=productMol,
        )
        reactionGraph.add_edge(reactionNode, moleculeNode)

print(f"Graph nodes: {reactionGraph.number_of_nodes():,}")
print(f"Graph edges: {reactionGraph.number_of_edges():,}")

Graph nodes: 218,151
Graph edges: 529,678


## 5. Select target products

In [21]:
terminalProductDF = moleculeDF[moleculeDF["isOnlyProduct"]].copy()
terminalProductDF = terminalProductDF.sort_values("numAppearances", ascending=False)

terminalProductDF.to_csv(os.path.join(reactionResultsDir, "terminalProductDF.csv"), index=False)

print(f"Terminal product candidates: {len(terminalProductDF):,}")
terminalProductDF.head(20)

Terminal product candidates: 59,761


,canonicalSmiles,numAppearances,numReactions,numAsReactant,numAsProduct,numStarterSources,exampleRule,isOnlyReactant,isOnlyProduct,isIntermediate
328,C=O,1458,1456,0,1458,13,rule0053_20,False,True,False
2,C,555,555,0,555,12,rule0126_2,False,True,False
59921,O=P(O)(O)OP(=O)(O)OP(=O)(O)O,133,133,0,133,2,rule0768_2,False,True,False
60678,O=P1(O)OP(=O)(O)O1,116,116,0,116,6,rule0549_1,False,True,False
25072,CCC(O)O,71,71,0,71,5,rule0078_14,False,True,False
25654,CCCO,69,69,0,69,5,rule0768_2,False,True,False
50988,O=C1COP(=O)(O)O1,68,68,0,68,2,rule0549_1,False,True,False
14983,CC(O)C(O)O,57,57,0,57,5,rule0125_5,False,True,False
11376,CC(C)(O)O,49,49,0,49,5,rule0043_12,False,True,False
13909,CC(O)(O)CO,48,48,0,48,5,rule0125_5,False,True,False


In [23]:
targetProductSmilesList = terminalProductDF["canonicalSmiles"].head(20).tolist()
targetProductSmilesList

['C=O',
 'C',
 'O=P(O)(O)OP(=O)(O)OP(=O)(O)O',
 'O=P1(O)OP(=O)(O)O1',
 'CCC(O)O',
 'CCCO',
 'O=C1COP(=O)(O)O1',
 'CC(O)C(O)O',
 'CC(C)(O)O',
 'CC(O)(O)CO',
 'O=CCl',
 'CC(=O)C(O)O',
 'O=C(O)COP(=O)(O)O',
 'CC(O)Cl',
 'O=CCC(=O)C(=O)O',
 'CC1OP(=O)(O)O1',
 'CC(O)(O)O',
 'COC(C)O',
 'O=P(O)(O)OCCO',
 'CCOC(=O)O']

### 6. Select starter molecules

In [24]:
starterMoleculeDF = moleculeDF[moleculeDF["isOnlyReactant"]].copy()
starterMoleculeDF = starterMoleculeDF.sort_values("numAppearances", ascending=False)

starterMoleculeSet = set(starterMoleculeDF["canonicalSmiles"])

starterMoleculeDF.to_csv(os.path.join(reactionResultsDir, "starterMoleculeDF.csv"), index=False)

print(f"Starter molecule candidates: {len(starterMoleculeDF):,}")
starterMoleculeDF

Starter molecule candidates: 2


,canonicalSmiles,numAppearances,numReactions,numAsReactant,numAsProduct,numStarterSources,exampleRule,isOnlyReactant,isOnlyProduct,isIntermediate
11554,CC(C)=CCOP(=O)(O)OP(=O)(O)O,5,5,5,0,1,rule0429_1,True,False,False
40780,F,1,1,1,0,1,rule1125_1,True,False,False


### Now search backward from each target product to a starter molecule

In [25]:
reverseReactionGraph = reactionGraph.reverse(copy=True)

reactionIdToRowDict = {
    row["reactionId"]: row.to_dict()
    for _, row in validReactionDF.iterrows()
}


def pathToReactionStepRecords(pathNodeList, routeId, targetSmiles):
    reactionNodes = [
        node for node in pathNodeList
        if str(node).startswith("reaction::")
    ]

    # Path is target → reaction → precursor in reverse graph.
    # Reverse reaction nodes to get forward biosynthetic order.
    reactionNodes = list(reversed(reactionNodes))

    stepRecords = []

    for stepIdx, reactionNode in enumerate(reactionNodes, start=1):
        reactionId = reactionNode.replace("reaction::", "")
        reactionRecord = reactionIdToRowDict[reactionId]

        stepRecords.append({
            "routeId": routeId,
            "stepId": stepIdx,
            "targetSmiles": targetSmiles,
            "reactionId": reactionId,
            "ruleName": reactionRecord["ruleName"],
            "reactionType": reactionRecord["reactionType"],
            "reactants": reactionRecord["canonicalReactants"],
            "products": reactionRecord["canonicalProducts"],
            "reactionString": reactionRecord["reactionString"],
            "thermo": reactionRecord["thermo"],
            "SourceStarterNum": reactionRecord["SourceStarterNum"],
            "SourceDirectory": reactionRecord["SourceDirectory"],
        })

    return stepRecords


routeStepRecords = []
routeSummaryRecords = []

maxRoutesPerTarget = 10
maxPathLength = 12

routeCounter = 0

starterNodeList = [
    f"molecule::{smiles}"
    for smiles in starterMoleculeSet
    if f"molecule::{smiles}" in reverseReactionGraph
]

for targetSmiles in targetProductSmilesList:
    targetNode = f"molecule::{targetSmiles}"

    if targetNode not in reverseReactionGraph:
        continue

    foundRoutesForTarget = 0

    for starterNode in starterNodeList:
        if foundRoutesForTarget >= maxRoutesPerTarget:
            break

        try:
            pathNodeList = nx.shortest_path(
                reverseReactionGraph,
                source=targetNode,
                target=starterNode,
            )

            if len(pathNodeList) > maxPathLength:
                continue

            routeCounter += 1
            routeId = f"route_{routeCounter:06d}"

            stepRecords = pathToReactionStepRecords(
                pathNodeList=pathNodeList,
                routeId=routeId,
                targetSmiles=targetSmiles,
            )

            if not stepRecords:
                continue

            routeStepRecords.extend(stepRecords)

            routeSummaryRecords.append({
                "routeId": routeId,
                "targetSmiles": targetSmiles,
                "starterSmiles": starterNode.replace("molecule::", ""),
                "numSteps": len(stepRecords),
                "pathLengthNodes": len(pathNodeList),
            })

            foundRoutesForTarget += 1

        except nx.NetworkXNoPath:
            continue

routeStepDF = pd.DataFrame(routeStepRecords)
routeSummaryDF = pd.DataFrame(routeSummaryRecords)

routeStepDF.to_csv(os.path.join(reactionResultsDir, "routeStepDF.csv"), index=False)
routeSummaryDF.to_csv(os.path.join(reactionResultsDir, "routeSummaryDF.csv"), index=False)

print(f"Routes found: {len(routeSummaryDF):,}")
print(f"Route ste  ps : {len(routeStepDF):,}")

routeSummaryDF.head()

Routes found: 40
Route steps : 98


,routeId,targetSmiles,starterSmiles,numSteps,pathLengthNodes
0,route_000001,C=O,F,2,5
1,route_000002,C=O,CC(C)=CCOP(=O)(O)OP(=O)(O)O,3,7
2,route_000003,C,F,3,7
3,route_000004,C,CC(C)=CCOP(=O)(O)OP(=O)(O)O,3,7
4,route_000005,O=P(O)(O)OP(=O)(O)OP(=O)(O)O,F,2,5


### Merge routes with enzyme annotations

## 3. Build molecule table and optional canonical SMILES

In [6]:
def canonicalizeSmiles(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    return Chem.MolToSmiles(mol, canonical=True) if mol is not None else None

moleculeRecords = []
for smiles in sorted(uniqueReactantMolecules | uniqueProductMolecules):
    moleculeRecords.append({
        "SMILES": smiles,
        "Canonical_SMILES": canonicalizeSmiles(smiles),
        "appearsAsReactant": smiles in uniqueReactantMolecules,
        "appearsAsProduct": smiles in uniqueProductMolecules,
    })

moleculeDF = pd.DataFrame(moleculeRecords)
moleculeDF.to_csv(DNADesignResultsDir + "moleculeDF.csv", index=False)
moleculeDF.head()

,SMILES,Canonical_SMILES,appearsAsReactant,appearsAsProduct
0,*C1=C(*)C(=O)C(*)=C(*)C1=O,*C1=C(*)C(=O)C(*)=C(*)C1=O,True,True
1,*c1c(*)c(O)c(*)c(*)c1O,*c1c(*)c(O)c(*)c(*)c1O,True,True
2,C,C,False,True
3,C#N,C#N,True,True
4,C1CO1,C1CO1,False,True


## 4. Choose route-step input

Use your curated recommended pathway table if available. Otherwise, the parsed reaction network is used as a step-level design database.

In [ ]:
# Build route-step table directly from parsed DORAnet reaction network

routeStepDF = reactionDF.copy()

routeStepDF["routeId"] = "network_" + routeStepDF["SourceDirectory"].astype(str)
routeStepDF["stepId"]  = routeStepDF.groupby("routeId").cumcount() + 1

requiredCols = [
    "routeId",
    "stepId",
    "reactants",
    "products",
    "reactionString",
    "ruleName",
    "reactionType",
    "thermo",
    "reactantStoich",
    "productStoich",
    "SourceStarterNum",
    "SourceDirectory",
]

for col in requiredCols:
    if col not in routeStepDF.columns:
        routeStepDF[col] = ""

routeStepDF = routeStepDF[requiredCols].copy()

routeStepDF.to_csv(DNADesignResultsDir + "routeStepDF.csv", index=False)

print(f"Route step rows: {len(routeStepDF):,}")
routeStepDF.head()

## 5. Rank routes if potency / toxicity / ADME columns are present

In [ ]:
reference_compounds = (
    pd.read_csv(resultsDir + "/Top20_InoAdeGuoXan_DORAnetGenerated_ARTprediction.csv")
    .nlargest(20, "pPotency_prediction")
    .reset_index(drop=True)
)
reference_compounds

In [ ]:
def minMax01(series, higherIsBetter=True):
    values = pd.to_numeric(series, errors="coerce")
    if values.notna().sum() == 0 or values.max() == values.min():
        return pd.Series(np.nan, index=series.index)

    scaled = (values - values.min()) / (values.max() - values.min())
    return scaled if higherIsBetter else 1 - scaled


# --------------------------------------------------
# Filter route steps using reference_compounds
# --------------------------------------------------

referenceSmilesSet = set(
    reference_compounds["Canonical_SMILES"]
    .dropna()
    .astype(str)
    .str.strip()
)

routeStepFilteredDF = routeStepDF.copy()

if "Canonical_SMILES" in routeStepFilteredDF.columns:
    routeStepFilteredDF["Canonical_SMILES"] = (
        routeStepFilteredDF["Canonical_SMILES"]
        .astype(str)
        .str.strip()
    )

    routeStepFilteredDF = routeStepFilteredDF[
        routeStepFilteredDF["Canonical_SMILES"].isin(referenceSmilesSet)
    ].copy()

else:
    # Fallback: use individual product molecules from DORAnet products column
    routeStepFilteredDF["productMoleculeList"] = routeStepFilteredDF["products"].apply(splitMoleculeString)

    routeStepFilteredDF = routeStepFilteredDF.explode("productMoleculeList").reset_index(drop=True)
    routeStepFilteredDF["Canonical_SMILES"] = (
        routeStepFilteredDF["productMoleculeList"]
        .astype(str)
        .str.strip()
    )

    routeStepFilteredDF = routeStepFilteredDF[
        routeStepFilteredDF["Canonical_SMILES"].isin(referenceSmilesSet)
    ].copy()

    routeStepFilteredDF = routeStepFilteredDF.drop(columns=["productMoleculeList"])


print(f"Original route-step rows : {len(routeStepDF):,}")
print(f"Filtered route-step rows : {len(routeStepFilteredDF):,}")
print(f"Reference compounds used : {len(referenceSmilesSet):,}")


# --------------------------------------------------
# Score only filtered route steps
# --------------------------------------------------

routeScoreDF = routeStepFilteredDF.groupby("routeId").agg(
    routeLength=("stepId", "count"),
    finalProduct=("products", "last"),
    finalCanonicalSMILES=("Canonical_SMILES", "last"),
).reset_index()


if "pPotency_prediction" in routeStepFilteredDF.columns:
    potencyByRoute = routeStepFilteredDF.groupby("routeId")["pPotency_prediction"].max()
    routeScoreDF = routeScoreDF.merge(
        potencyByRoute.rename("bestPotency").reset_index(),
        on="routeId",
        how="left"
    )
    routeScoreDF["potencyScore01"] = minMax01(routeScoreDF["bestPotency"], True)


if "coreToxicityScore" in routeStepFilteredDF.columns:
    toxByRoute = routeStepFilteredDF.groupby("routeId")["coreToxicityScore"].min()
    routeScoreDF = routeScoreDF.merge(
        toxByRoute.rename("bestToxicity").reset_index(),
        on="routeId",
        how="left"
    )
    routeScoreDF["toxicityScore01"] = minMax01(routeScoreDF["bestToxicity"], False)


if "ADMEFeasibility" in routeStepFilteredDF.columns:
    admeByRoute = routeStepFilteredDF.groupby("routeId")["ADMEFeasibility"].max()
    routeScoreDF = routeScoreDF.merge(
        admeByRoute.rename("bestADME").reset_index(),
        on="routeId",
        how="left"
    )
    routeScoreDF["admeScore01"] = minMax01(routeScoreDF["bestADME"], True)


scoreCols = [
    col for col in ["potencyScore01", "toxicityScore01", "admeScore01"]
    if col in routeScoreDF.columns
]

if scoreCols:
    routeScoreDF["routePriorityScore"] = routeScoreDF[scoreCols].mean(axis=1)
else:
    routeScoreDF["routePriorityScore"] = 1 / routeScoreDF["routeLength"].clip(lower=1)


routeScoreDF = (
    routeScoreDF
    .sort_values("routePriorityScore", ascending=False)
    .reset_index(drop=True)
)


# Save outputs
routeStepFilteredDF.to_csv(DNADesignResultsDir + "routeStepFilteredDF.csv", index=False)
routeScoreDF.to_csv(DNADesignResultsDir + "routeScoreDF.csv", index=False)

routeScoreDF.head(10)

## 6. Map reaction steps to enzyme hypotheses

In [ ]:
enzymeRuleMap = [
    (["methyl", "methylation"], "methyltransferase", "2.1.1.-", "SAM"),
    (["glycosyl", "glucosyl", "sugar"], "glycosyltransferase", "2.4.-.-", "UDP-sugar"),
    (["phosph", "kinase"], "kinase / phosphotransferase", "2.7.-.-", "ATP"),
    (["oxid", "hydroxyl", "monooxygenase"], "oxidoreductase / monooxygenase", "1.14.-.-", "NAD(P)H, O2"),
    (["reduct", "dehydrogen"], "reductase / dehydrogenase", "1.-.-.-", "NAD(P)H"),
    (["hydrolysis", "hydrolase"], "hydrolase", "3.-.-.-", "H2O"),
    (["amide", "acyl", "ligase"], "ligase / acyltransferase", "6.-.-.- or 2.3.-.-", "ATP or acyl-CoA"),
    (["transamin", "amination", "amine"], "aminotransferase", "2.6.-.-", "PLP"),
    (["carbox", "decarbox"], "carboxylase / decarboxylase", "4.1.-.- or 6.4.-.-", "CO2 / biotin / ATP"),
]

def inferEnzymeHypothesis(ruleName, reactionType):
    text = f"{ruleName} {reactionType}".lower()
    for keywords, enzymeClass, ecHint, cofactorHint in enzymeRuleMap:
        if any(keyword in text for keyword in keywords):
            return enzymeClass, ecHint, cofactorHint, "keyword_match"
    return "manual_review_required", "unknown", "unknown", "no_keyword_match"

def buildEnzymeSearchQuery(row):
    return " | ".join([
        f"enzyme class: {row['enzymeClass']}",
        f"EC hint: {row['ecHint']}",
        f"reaction type: {row['reactionType']}",
        f"rule: {row['ruleName']}",
        f"reactants: {str(row['reactants'])[:120]}",
        f"products: {str(row['products'])[:120]}",
    ])

enzymeHypothesisDF = routeStepDF.copy()
hypothesisRows = enzymeHypothesisDF.apply(
    lambda row: inferEnzymeHypothesis(row.get("ruleName", ""), row.get("reactionType", "")), axis=1
)
enzymeHypothesisDF[["enzymeClass", "ecHint", "cofactorHint", "inferenceMethod"]] = pd.DataFrame(
    hypothesisRows.tolist(), index=enzymeHypothesisDF.index
)
enzymeHypothesisDF["enzymeSearchQuery"] = enzymeHypothesisDF.apply(buildEnzymeSearchQuery, axis=1)
enzymeHypothesisDF["manualReviewRequired"] = enzymeHypothesisDF["enzymeClass"].eq("manual_review_required")

enzymeHypothesisDF.to_csv(DNADesignResultsDir + "enzymeHypothesisDF.csv", index=False)
enzymeHypothesisDF[["routeId", "stepId", "reactionType", "ruleName", "enzymeClass", "ecHint", "cofactorHint", "manualReviewRequired"]].head(10)

## 7. Create enzyme candidate template

Fill this CSV after searching UniProt, BRENDA, Rhea, KEGG, MetaCyc, and literature. Keep one row per candidate enzyme per pathway step.

In [ ]:
candidateTemplateCols = [
    "routeId", "stepId", "reactants", "products", "reactionType", "ruleName",
    "enzymeClass", "ecHint", "cofactorHint", "enzymeSearchQuery",
]

enzymeCandidateTemplateDF = enzymeHypothesisDF[candidateTemplateCols].copy()

enzymeCandidateTemplateDF["enzymeName"] = "TO_FILL"
enzymeCandidateTemplateDF["ecNumber"] = "TO_FILL"
enzymeCandidateTemplateDF["sourceOrganism"] = "TO_FILL"
enzymeCandidateTemplateDF["proteinAccession"] = "TO_FILL"
enzymeCandidateTemplateDF["proteinSequenceAvailable"] = False
enzymeCandidateTemplateDF["evidenceNotes"] = "TO_FILL"
enzymeCandidateTemplateDF["evidenceScore"] = np.nan
enzymeCandidateTemplateDF["substrateScopeScore"] = np.nan
enzymeCandidateTemplateDF["hostCompatibilityScore"] = np.nan
enzymeCandidateTemplateDF["expressionPrecedentScore"] = np.nan
enzymeCandidateTemplateDF["biosecurityFlag"] = "review_required"
enzymeCandidateTemplateDF["selectedForDesign"] = False

templatePath = DNADesignResultsDir + "enzymeCandidateTemplate.csv"
enzymeCandidateTemplateDF.to_csv(templatePath, index=False)
print(templatePath)
enzymeCandidateTemplateDF.head()

## 8. Rank filled enzyme candidates

In [ ]:
scoreWeights = {
    "evidenceScore": 0.35,
    "substrateScopeScore": 0.30,
    "hostCompatibilityScore": 0.20,
    "expressionPrecedentScore": 0.15,
}

if Path(enzymeCandidatesCsv).exists():
    enzymeCandidateDF = pd.read_csv(enzymeCandidatesCsv)
    for col in scoreWeights:
        enzymeCandidateDF[col] = pd.to_numeric(enzymeCandidateDF.get(col, 0), errors="coerce").fillna(0)

    enzymeCandidateDF["enzymeConfidenceScore"] = sum(
        scoreWeights[col] * enzymeCandidateDF[col] for col in scoreWeights
    )
    enzymeCandidateDF["blockedByBiosecurityFlag"] = enzymeCandidateDF["biosecurityFlag"].astype(str).str.lower().ne("clear")

    selectedEnzymeDF = (
        enzymeCandidateDF
        .query("blockedByBiosecurityFlag == False")
        .sort_values(["routeId", "stepId", "enzymeConfidenceScore"], ascending=[True, True, False])
        .groupby(["routeId", "stepId"], as_index=False)
        .head(maxEnzymeCandidatesPerStep)
        .reset_index(drop=True)
    )
else:
    enzymeCandidateDF = enzymeCandidateTemplateDF.copy()
    enzymeCandidateDF["enzymeConfidenceScore"] = np.nan
    selectedEnzymeDF = enzymeCandidateDF.copy()

if not selectedEnzymeDF.empty:
    selectedEnzymeDF["candidateRank"] = selectedEnzymeDF.groupby(["routeId", "stepId"]).cumcount() + 1

selectedEnzymeDF.to_csv(DNADesignResultsDir + "selectedEnzymeDF.csv", index=False)
selectedEnzymeDF.head()

## 9. Generate non-operational DNA design plan

This table records the expression-construct strategy for review. It intentionally does not generate synthesis-ready DNA sequences.

In [ ]:
def promoterPlanForHost(hostSystem):
    if "cell_free" in hostSystem.lower():
        return "cell-free-compatible expression control; tune after single-enzyme test"
    if "coli" in hostSystem.lower():
        return "medium inducible bacterial promoter; tune by promoter/RBS library"
    if "yeast" in hostSystem.lower():
        return "yeast-compatible promoter; tune by promoter copy/strength"
    return "host-appropriate promoter; tune experimentally"

def vectorPlanForStep(stepCount):
    if stepCount <= 1:
        return "single-enzyme validation construct"
    if stepCount <= 3:
        return "modular multi-enzyme construct after single-step validation"
    return "split pathway into modules before full assembly"

routeLengthMap = routeStepDF.groupby("routeId")["stepId"].count().to_dict()

dnaDesignRecords = []
for _, row in selectedEnzymeDF.iterrows():
    routeId = row["routeId"]
    stepId = row["stepId"]
    candidateRank = int(row.get("candidateRank", 1)) if pd.notna(row.get("candidateRank", 1)) else 1
    routeLength = routeLengthMap.get(routeId, np.nan)

    dnaDesignRecords.append({
        "constructId": f"{routeId}_step{stepId}_cand{candidateRank}",
        "routeId": routeId,
        "stepId": stepId,
        "candidateRank": candidateRank,
        "hostSystem": hostSystem,
        "enzymeName": row.get("enzymeName", "TO_FILL"),
        "proteinAccession": row.get("proteinAccession", "TO_FILL"),
        "enzymeClass": row.get("enzymeClass", ""),
        "cofactorHint": row.get("cofactorHint", ""),
        "constructPurpose": "enzyme expression feasibility review",
        "promoterPlan": promoterPlanForHost(hostSystem),
        "fivePrimeControlPlan": "host-appropriate RBS/5'UTR; tune after enzyme choice",
        "codingSequencePolicy": "retrieve native CDS or codon-optimize only after biosafety and IP review",
        "terminatorPlan": "standard host-compatible terminator",
        "vectorPlan": vectorPlanForStep(routeLength),
        "sequenceGenerated": False,
        "synthesisReady": False,
        "reviewStatus": "biosafety_and_domain_expert_review_required",
        "notes": "Non-operational design record; no raw synthesis-ready DNA sequence generated.",
    })

dnaDesignPlanDF = pd.DataFrame(dnaDesignRecords)
dnaDesignPlanDF.to_csv(DNADesignResultsDir + "dnaDesignPlanDF.csv", index=False)
dnaDesignPlanDF.head()

## 10. Write route design cards

In [ ]:
def compactText(value, width=90):
    return textwrap.shorten(str(value), width=width, placeholder="...")

cardLines = ["# DORAnet downstream DNA design cards", ""]

for routeId, stepGroupDF in enzymeHypothesisDF.groupby("routeId"):
    scoreRowDF = routeScoreDF.query("routeId == @routeId")
    scoreText = "not scored"
    if not scoreRowDF.empty:
        scoreText = f"priority={scoreRowDF.iloc[0]['routePriorityScore']:.3f}; length={scoreRowDF.iloc[0]['routeLength']}"

    cardLines += [f"## {routeId}", f"- Route score: {scoreText}", ""]
    for _, stepRow in stepGroupDF.sort_values("stepId").iterrows():
        cardLines += [
            f"### Step {stepRow['stepId']}",
            f"- Reaction: `{compactText(stepRow['reactionString'])}`",
            f"- DORAnet rule: `{compactText(stepRow['ruleName'])}`",
            f"- Enzyme hypothesis: **{stepRow['enzymeClass']}**; EC hint `{stepRow['ecHint']}`; cofactor `{stepRow['cofactorHint']}`",
            f"- Search query: {compactText(stepRow['enzymeSearchQuery'], 160)}",
            "",
        ]

    designRowsDF = dnaDesignPlanDF.query("routeId == @routeId") if not dnaDesignPlanDF.empty else pd.DataFrame()
    if not designRowsDF.empty:
        cardLines += ["### DNA design records", ""]
        for _, designRow in designRowsDF.iterrows():
            cardLines += [
                f"- `{designRow['constructId']}`: {designRow['constructPurpose']}; synthesisReady={designRow['synthesisReady']}; review={designRow['reviewStatus']}",
            ]
        cardLines.append("")

cardsPath = DNADesignResultsDir + "routeDesignCards.md"
cardsPath.write_text("
".join(cardLines), encoding="utf-8")
print(cardsPath)

## 11. Outputs

Core files generated in `DNADesignResultsDir`:

- `reactionDF.csv`: parsed DORAnet reactions
- `moleculeDF.csv`: unique reactant/product molecules
- `routeStepDF.csv`: pathway-step table
- `routeScoreDF.csv`: route-level prioritization
- `enzymeHypothesisDF.csv`: reaction-to-enzyme hypotheses
- `enzymeCandidateTemplate.csv`: template for enzyme database/literature review
- `selectedEnzymeDF.csv`: ranked enzyme candidates if a filled template is provided
- `dnaDesignPlanDF.csv`: non-operational expression-design records
- `routeDesignCards.md`: concise route cards for review